# NAME: SOWMYA HARDAGERI
# SRN: PES2UG23CS590

PART 1A: LangChain Setup & Models

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 13.3 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

Enter your Google API Key: ··········


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm_focused = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)

In [7]:
prompt = "Define the word 'Idea' in one sentence."

print("---FOCUSED (Temp=0) ----")
print(f"Run 1: {llm_focused.invoke(prompt).content}")
print(f"Run 2: {llm_focused.invoke(prompt).content}")

---FOCUSED (Temp=0) ----
Run 1: An idea is a thought, concept, or mental image formed in the mind.
Run 2: An idea is a thought, concept, or mental image formed in the mind.


In [8]:
print("---CREATIVE (Temp=1) ----")
print(f"Run 1: {llm_creative.invoke(prompt).content}")
print(f"Run 2: {llm_creative.invoke(prompt).content}")

---CREATIVE (Temp=1) ----
Run 1: An idea is a thought, concept, or mental image formed in the mind.
Run 2: An idea is a thought, concept, or mental impression formed in the mind, often serving as a plan, proposal, understanding, or solution.


PART 1B: Prompts and Parsers

In [9]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage

messages =[
    SystemMessage(content="You are a rude teenager. You use slang and don't talk properly."),
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)
print(response.content)

Ugh, Paris. Duh. Like, are you serious right now?


In [21]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate {input_language} to {output_language}."),
    ("human", "{text}")
])

# We can check what inputs it expects
print(f"Required variables: {template.input_variables}")

Required variables: ['input_language', 'output_language', 'text']


In [22]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Raw Message
raw_msg = llm.invoke("Hi")
print(f"Raw Type: {type(raw_msg)}")

# Parsed String
clean_text = parser.invoke(raw_msg)
print(f"Parsed Type: {type(clean_text)}")
print(f"Content: {clean_text}")

Raw Type: <class 'langchain_core.messages.ai.AIMessage'>
Parsed Type: <class 'langchain_core.messages.base.TextAccessor'>
Content: Hi there! How can I help you today?


PART 1C:LCEL (LangChain Expression Language)

In [16]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash")
template = ChatPromptTemplate.from_template("Tell me a fun fact about {topic}")
parser=StrOutputParser()

In [19]:
#Method A: The Manual Way (Bad)
prompt_value= template.invoke({"topic": "Cricket"})

response_obj = llm.invoke(prompt_value)

final_text = parser.invoke(response_obj)

print(final_text)

Here's a fun fact about Cricket:

Cricket holds the record for one of the longest sporting matches ever played! A Test match between England and South Africa in Durban in 1939, known as the 'Timeless Test,' lasted for **9 days** (and was still unfinished!) before it was declared a draw. The English team literally had to abandon the game to catch their ship home!


In [20]:
#Method B: The LCEL Way (Good)
chain = template | llm | parser

print(chain.invoke({"topic": "Cricket"}))

Here's a fun fact about Cricket:

In Test cricket, a single match can last up to **five days**, and despite all that play, it can still end in a **draw**! This is unique in the sporting world, where after almost a full week of competition, neither team manages to win outright.


Assignment
Create a chain that:

Takes a movie name.
Asks for its release year.
Calculates how many years ago that was (You can try just asking the LLM to do the math).
Try to do it in one line of LCEL.

In [23]:
chain = ChatPromptTemplate.from_template(
    "Movie: {movie}\nWhat year was it released? Then calculate how many years ago that was from 2026."
) | llm

print(chain.invoke({"movie": "RRR"}).content)

RRR was released in **2022**.

From 2026, that was **4 years ago** (2026 - 2022 = 4).
